# Estimating the number of microlensing events

The goal of this notebook is to explore the number of Bulge events to be selected in the Key Project, and to estimate the observing time required. 

This estimate is based on the catalog of events alerted by OGLE & KMTNet from the Bulge region. 

In [2]:
from os import path, getcwd
import csv
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table, Column
from astropy.io import fits
import numpy as np
import pandas as pd
import h5py
from sys import path as pythonpath
pythonpath.append('../')
import matplotlib.pyplot as plt

## Bulge Events

We estimate the expected numbers of events based on OGLE and KMTNet catalogs, since as optical surveys their targets are more accessible than PRIME's NIR ones.  

In [23]:
def read_event_catalog(file_path):
    column_types = {
        'Name': 'str',
        'RA': 'str',
        'Dec': 'str', 
        't0[HJD]': 'float',
        'tE[days]': 'float',
        'u0': 'float',
        'Ibase': 'float',
        'Ibase_err': 'float'
    }
    df = pd.read_table(file_path, sep='\s+', dtype=column_types)
    
    return df

<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/1d/5hlyfsgd0kl_nd815xmp9cv00000gn/T/ipykernel_88454/900504522.py:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df = pd.read_table(file_path, sep='\s+', dtype=column_types)


OGLE and MOA survey overlapping regions of the sky and thereby tend to detect overlapping sets of events.  While MOA do detect some events that OGLE don't and vice versa, the OGLE catalog is more extensive, and more easily queried programmatically, so we use that as a representative sample for the basis of this estimate. 

In [24]:
event_file_path = path.join(getcwd(), '../data', 'ogle_catalog.dat')
ogle_events = read_event_catalog(event_file_path)
ogle_events

,Name,RA,Dec,t0[HJD],tE[days],u0,Ibase,Ibase_err
0,OGLE-1998-BUL-01,17:53:40.36,-30:10:20.1,2450887.185,39.770,0.306,17.180,NaN
1,OGLE-1998-BUL-02,18:00:40.20,-29:19:35.2,2450890.374,58.644,0.371,18.042,NaN
2,OGLE-1998-BUL-03,18:01:18.12,-29:05:49.9,2450901.185,45.860,0.587,17.034,NaN
3,OGLE-1998-BUL-04,18:02:29.02,-28:33:12.5,2450913.623,17.332,0.510,17.227,NaN
4,OGLE-1998-BUL-05,17:48:42.80,-35:23:08.5,2450914.155,19.154,0.152,18.337,NaN
...,...,...,...,...,...,...,...,...
25487,OGLE-2025-BLG-1494,17:35:12.78,-29:30:11.6,2460966.828,15.650,1.259,16.157,16.157
25488,OGLE-2025-BLG-1495,17:42:30.66,-24:43:58.0,2461004.837,86.290,0.175,18.715,18.715
25489,OGLE-2025-BLG-1496,17:12:20.69,-31:31:56.9,2460970.666,17.520,1.054,14.279,14.279
25490,OGLE-2025-BLG-1497,17:45:06.27,-34:17:25.1,2455655.555,13.000,0.100,18.000,18.000


In [30]:
event_file_path = path.join(getcwd(), '../data', 'kmtnet_catalog.dat')
kmtnet_events = read_event_catalog(event_file_path)
kmtnet_events

,Name,RA,Dec,t0[HJD],tE[days],u0,Ibase,Ibase_err
0,KMT-2016-BLG-0001,17:54:14.09,-28:40:58.91,247549.19296,39.10,0.704,19.04,NaN
1,KMT-2016-BLG-0002,17:54:09.09,-28:25:06.13,247533.68206,6.22,1.000,17.52,NaN
2,KMT-2016-BLG-0003,17:54:03.78,-28:26:34.37,247569.58472,37.21,0.325,17.91,NaN
3,KMT-2016-BLG-0004,17:53:29.68,-28:40:30.68,247567.36853,25.68,0.256,18.93,NaN
4,KMT-2016-BLG-0005,17:53:06.41,-28:36:52.02,247522.98201,141.71,0.002,20.00,NaN
...,...,...,...,...,...,...,...,...
27698,KMT-2025-BLG-2600,17:47:44.00,-34:24:46.51,60945.00254,51.94,0.286,18.52,NaN
27699,KMT-2025-BLG-2601,17:44:18.73,-36:08:00.67,60946.40782,29.64,0.171,18.68,NaN
27700,KMT-2025-BLG-2602,17:55:35.97,-32:04:20.32,60951.72433,61.96,0.513,19.29,NaN
27701,KMT-2025-BLG-2603,18:00:03.82,-27:14:11.00,60944.54399,4.58,0.121,19.14,NaN


For future reference add columns with the magnification A for each event to the event tables.

In [31]:
def calc_mag(u):
    A = (u*u + 2) / (u * np.sqrt(u*u + 4))
    return A

In [34]:
ogle_events['A0'] = calc_mag(ogle_events['u0'])
kmtnet_events['A0'] = calc_mag(kmtnet_events['u0'])

### Number of High-Mag Events

The first science goal is to observe events with $A_{max} > 80$, so we can use the existing catalogs of events to date to estimate the frequency of high mag events per year. An IBase threshold gives control over the selection of events that are bright enough for LCO to observe. 

In [104]:
def count_events_per_year(event_set, ibase_thresh=18.5, Athresh=None):
    events_per_year = {}
    for index, event in event_set.iterrows():
        if event['Ibase'] <= ibase_thresh:
            name = str(event['Name'])
            if '-' in name:
                year = name.split('-')[1]
            else:
                year = int('20' + name[4:6])
            add_event = True
            if Athresh:
                if event['A0'] < Athresh:
                    add_event = False 
            if add_event:
                if year not in events_per_year.keys():
                    events_per_year[year] = [event['Name']]
                else:
                    events_per_year[year].append(event['Name'])

    for key, event_list in events_per_year.items():
        print(str(key)+': '+str(len(event_list))+' events')
    return events_per_year


In [100]:
print('Counts of all OGLE events per year:')
ogle_events_per_year = count_events_per_year(ogle_events)

print('OGLE high-mag events per year:')
ogle_HM_events_per_year = count_events_per_year(ogle_events, Athresh=80)

Counts of all OGLE events per year:
1998: 33 events
1999: 35 events
2000: 63 events
2002: 196 events
2003: 258 events
2004: 349 events
2005: 352 events
2006: 299 events
2007: 319 events
2008: 356 events
2009: 108 events
2011: 715 events
2012: 718 events
2013: 839 events
2014: 836 events
2015: 858 events
2016: 882 events
2017: 753 events
2018: 794 events
2019: 699 events
2022: 200 events
2023: 718 events
2024: 733 events
2025: 778 events
OGLE high-mag events per year:
1998: 2 events
2000: 5 events
2002: 18 events
2003: 20 events
2004: 29 events
2005: 22 events
2006: 18 events
2007: 17 events
2008: 15 events
2009: 8 events
2011: 28 events
2012: 27 events
2013: 31 events
2014: 42 events
2015: 37 events
2016: 41 events
2017: 34 events
2018: 43 events
2019: 19 events
2022: 18 events
2023: 29 events
2024: 30 events
2025: 34 events


In [90]:
print('Percentage of high-mag events per year: ')
sum = 0.0
count = 0.0
for year, all_events in ogle_events_per_year.items(): 
    if year in ogle_HM_events_per_year.keys():
        hm_events = ogle_HM_events_per_year[year]
        percent = (len(hm_events)/len(all_events))*100.0
        print(year + ': ' + str(round(percent,1)) + '%')
        sum += percent
        count += 1.0
    else:
        print(year + ': 0.0%')
ogle_hm_percent = sum/count
print('Average percentage of high mag events: ' + str(round((sum/count),1)) + '%')

Percentage of high-mag events per year: 
1998: 6.1%
1999: 0.0%
2000: 7.9%
2002: 9.2%
2003: 7.8%
2004: 8.3%
2005: 6.2%
2006: 6.0%
2007: 5.3%
2008: 4.2%
2009: 7.4%
2011: 3.9%
2012: 3.8%
2013: 3.7%
2014: 5.0%
2015: 4.3%
2016: 4.6%
2017: 4.5%
2018: 5.4%
2019: 2.7%
2022: 9.0%
2023: 4.0%
2024: 4.1%
2025: 4.4%
Average percentage of high mag events: 5.6%


In [91]:
print('Counts of all KMTNet events per year:')
kmtnet_events_per_year = count_events_per_year(kmtnet_events)

print('KMTNet high-mag events per year:')
kmtnet_HM_events_per_year = count_events_per_year(kmtnet_events, Athresh=80)

Counts of all KMTNet events per year:
2016: 759 events
2017: 776 events
2018: 673 events
2019: 829 events
2020: 278 events
2021: 847 events
2022: 764 events
2023: 834 events
2024: 878 events
2025: 656 events
KMTNet high-mag events per year:
2016: 38 events
2017: 28 events
2018: 24 events
2019: 40 events
2020: 21 events
2021: 42 events
2022: 41 events
2023: 35 events
2024: 45 events
2025: 37 events


In [92]:
print('Percentage of high-mag events per year: ')
sum = 0.0
count = 0.0
for year, all_events in kmtnet_events_per_year.items(): 
    if year in kmtnet_HM_events_per_year.keys():
        hm_events = kmtnet_HM_events_per_year[year]
        percent = (len(hm_events)/len(all_events))*100.0
        print(year + ': ' + str(round(percent,1)) + '%')
        sum += percent
        count += 1.0
    else:
        print(year + ': 0.0%')
kmtnet_hm_percent = sum/count
print('Average percentage of high mag events: ' + str(round((sum/count),1)) + '%')

Percentage of high-mag events per year: 
2016: 5.0%
2017: 3.6%
2018: 3.6%
2019: 4.8%
2020: 7.6%
2021: 5.0%
2022: 5.4%
2023: 4.2%
2024: 5.1%
2025: 5.6%
Average percentage of high mag events: 5.0%


Since both OGLE and KMTNet cover magnitude ranges similar to that of LCO, this percentage of events can be used to estimate the annual number of high magnification targets. 

In [93]:
def mean_nevents(survey_events_per_year):
    nevents = 0
    for year, events in survey_events_per_year.items():
        nevents += len(events) 
    nevents_year = nevents / len(survey_events_per_year)
    print('Mean number of events per year = ' + str(round(nevents_year,0)))
    return nevents_year

In [94]:
mean_ogle_nevents_year = mean_nevents(ogle_events_per_year)
mean_kmtnet_nevents_year = mean_nevents(kmtnet_events_per_year)

Mean number of events per year = 495.0
Mean number of events per year = 729.0


In [1]:
hm_percent = (ogle_hm_percent + kmtnet_hm_percent) / 2.0
print('Estimate number of OGLE HM events per year: ' + str(round((mean_ogle_nevents_year * hm_percent/100.0),0)))
print('Estimate number of KMTNet HM events per year: ' + str(round((mean_kmtnet_nevents_year * hm_percent/100.0),0)))

NameError: name 'ogle_hm_percent' is not defined

Based on this analysis, we estimate the number of targets for this element of the program to be 30 per year.

## Plane events

The catalog of events from the Gaia Mission are the best available approximation of the targets we can expect to detect in the wider Galactic plane, from LSST, LS4, ZTF etc, and the magnitude range covered is a good approximation to the range that LCO can observe. 

In [97]:
event_file_path = path.join(getcwd(), '../data', 'gaia_catalog.dat')
gaia_events = read_event_catalog(event_file_path)
gaia_events

,Name,RA,Dec,t0[HJD],tE[days],u0,Ibase,Ibase_err
0,Gaia24dtt,15:10:58.3656,-55:25:56.784,NaN,NaN,NaN,18.00,NaN
1,Gaia24dqj,23:36:54.9336,+55:01:33.204,NaN,NaN,NaN,15.69,NaN
2,Gaia24dnp,19:55:11.244,+25:17:24.18,NaN,NaN,NaN,15.77,NaN
3,Gaia24dmu,19:03:01.9584,+18:44:10.284,NaN,NaN,NaN,15.97,NaN
4,Gaia24dme,19:17:24.0816,+12:20:03.66,NaN,NaN,NaN,15.78,NaN
...,...,...,...,...,...,...,...,...
689,Gaia17bbs,18:10:49.2288,-22:36:55.98,NaN,NaN,NaN,19.04,NaN
690,Gaia17ajg,17:57:50.0376,-31:33:18.18,NaN,NaN,NaN,18.11,NaN
691,Gaia17ahl,17:20:12.7176,-43:54:01.944,NaN,NaN,NaN,19.21,NaN
692,Gaia17ahk,17:13:34.8528,-39:00:05.04,NaN,NaN,NaN,18.29,NaN


Unfortunately we don't have event models for these events but we do have at least Ibase, so we can roughly select the number of accessible targets. 

In [106]:
print('Counts of all Gaia events per year:')
gaia_events_per_year = count_events_per_year(gaia_events)

Counts of all Gaia events per year:
2024: 77 events
2023: 106 events
2022: 142 events
2021: 78 events
2020: 62 events
2019: 32 events
2018: 11 events
2017: 6 events


Since Gaia was limited in its ability to observe crowded fields, we combine the Gaia catalog of events with those from OGLE to better represent the expected catalog across the whole Galactic plane.  Unfortunately, we can't selected based on event parameters, but we can use the OGLE data to estimate the proportions of events in different timescale categories.  

In [127]:
def select_events_by_timescale(df, te_min, te_max, ibase_thresh=21.5, umax=0.2):
    filtered_df = df[
                (df['Ibase'] < ibase_thresh) & 
                 (df['tE[days]'] >= te_min) & 
                 (df['tE[days]'] <= te_max) & 
                 (df['u0'] <= umax)
        ]
    return filtered_df

In [128]:
short_events = select_events_by_timescale(ogle_events, 0.0, 100.0) 
long_events = select_events_by_timescale(ogle_events, 100.0, 3000.0) 

Now we can count the occurrance per year. 

In [129]:
print('Numbers of short-timescale events per year: ')
short_events_per_year = count_events_per_year(short_events)

print('Numbers of long-timescale events per year: ')
long_events_per_year = count_events_per_year(long_events)

Numbers of short-timescale events per year: 
1998: 5 events
1999: 6 events
2000: 11 events
2002: 32 events
2003: 53 events
2004: 82 events
2005: 88 events
2006: 61 events
2007: 58 events
2008: 66 events
2009: 25 events
2011: 129 events
2012: 137 events
2013: 159 events
2014: 195 events
2015: 159 events
2016: 194 events
2017: 138 events
2018: 158 events
2019: 131 events
2022: 52 events
2023: 117 events
2024: 147 events
2025: 157 events
Numbers of long-timescale events per year: 
1999: 2 events
2000: 1 events
2002: 5 events
2003: 5 events
2004: 4 events
2005: 5 events
2006: 10 events
2007: 4 events
2008: 13 events
2009: 3 events
2011: 14 events
2012: 12 events
2013: 11 events
2014: 16 events
2015: 20 events
2016: 18 events
2017: 18 events
2018: 17 events
2019: 15 events
2022: 3 events
2023: 16 events
2024: 14 events
2025: 14 events


In [130]:
mean_short_nevents_year = mean_nevents(short_events_per_year)

Mean number of events per year = 98.0


In [131]:
mean_long_nevents_year = mean_nevents(long_events_per_year)

Mean number of events per year = 10.0


Based on RTModel analysis of previous years of detections, we expect the following fractions of binary, planetary and long-timescale events. 

In [133]:
binary_frac = 2.6/100.0
planet_frac = 0.5/100.0
bh_frac = 2/100.0

n_binary_per_year = int(round((mean_ogle_nevents_year * binary_frac),0))
n_planet_per_year = int(round((mean_ogle_nevents_year * planet_frac),0))
n_bh_per_year = int(round((mean_ogle_nevents_year * bh_frac),0))

print('Expect Bulge surveys to detect the following numbers of events per year with magnitudes observable by LCO:')
print('N stellar binaries = '+str(n_binary_per_year))
print('N planet binaries = '+str(n_planet_per_year))
print('N BH = '+str(n_bh_per_year))


Expect Bulge surveys to detect the following numbers of events per year with magnitudes observable by LCO:
N stellar binaries = 13
N planet binaries = 2
N BH = 10


Based on the kp_simulator simulation, LCO would be able to contribute follow-up observations for 42% of events discovered in the Bulge (largely limited by apparent magnitude for events, which suffer higher extinction than elsewhere in the sky). 


In [134]:
obs_frac = 1.0
n_years = 3

n_binary_obs_per_year = int(round((obs_frac * n_binary_per_year),0))
n_binary = n_binary_obs_per_year * n_years
n_planet_obs_per_year = int(round((obs_frac * n_planet_per_year),0))
n_planet = n_planet_obs_per_year * n_years
n_bh_obs_per_year = int(round((obs_frac * n_bh_per_year),0))
n_bh = n_bh_obs_per_year * n_years

print('Expect LCO to be able to observe the following numbers of events in the Extended Survey Zone:')
print('N stellar binaries = '+str(n_binary_obs_per_year)+' per year, and '+str(n_binary)+' in total')
print('N planetary binaries = '+str(n_planet_obs_per_year)+' per year, and '+str(n_planet)+' in total')
print('N stellar remnants = '+str(n_bh_obs_per_year)+' per year, and '+str(n_bh)+' in total')


Expect LCO to be able to observe the following numbers of events in the Extended Survey Zone:
N stellar binaries = 13 per year, and 39 in total
N planetary binaries = 2 per year, and 6 in total
N stellar remnants = 10 per year, and 30 in total
